#### Librerías

In [0]:
dbutils.library.restartPython()

In [0]:
import os, requests
import mlflow
from mlflow.deployments import get_deploy_client
from typing import List, Dict


#### Parámetros:

In [0]:
TABLE_ = 'bind_agent.docs.silver_excel'
# LLM_ENDPOINT = 'databricks-gemma-3-12b'
LLM_ENDPOINT = 'databricks-llama-4-maverick' 

#### 1.- Levanto la tabla y chequeo esquema:

In [0]:
spark.table(TABLE_).display()

In [0]:
spark.table(TABLE_).printSchema()

#### 2.- Defino helper 

Helper para que LLM no genere columnas nuevas 

In [0]:
def schema_text(table: str) -> str:
    fields = spark.table(table).schema.fields
    return "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in fields])

SCHEMA_TEXT = schema_text(TABLE_)

#### 3.- Defino función o tool SQL

In [0]:
# def sql_tool(sql: str) -> str:
#     df = spark.sql(sql)
#     return df.limit(20).toPandas().to_string(index=False)

In [0]:
def sql_tool(sql: str, n: int = 20) -> str:
    df = spark.sql(sql)
    return df.limit(n).toPandas().to_string(index=False)

# me aseguro de obtener un resultado
def run_sql_preview(sql: str, n: int = 20) -> dict:
    try:
        df = spark.sql(sql)
        has_rows = df.limit(1).count() > 0
        preview = df.limit(n).toPandas().to_string(index=False)
        return {'ok': True, 'has_rows': has_rows, 'preview': preview, 'error': None}
    except Exception as e:
        return {'ok': False, 'has_rows': False, 'preview': '', 'error': str(e)}

#### 4.- Genero consulta SQL con el LLM:

##### 4.1 - Función para llamar al LLM

In [0]:
def extract_chat_content(resp) -> str:
    if isinstance(resp, dict):
        d = resp
    else:
        try:
            d = resp.__dict__
        except Exception:
            d = resp

    # formato: choices[0].message.content
    try:
        return d['choices'][0]['message']['content']
    except Exception:
        pass

    # otros formatos
    try:
        return d['predictions'][0]['content']
    except Exception:
        pass

    return str(resp)

def call_llm(endpoint: str, messages: List[Dict[str, str]], temperature: float = 0.2, max_tokens: int = 500) -> str:
    payload = {'messages': messages, 'temperature': temperature, 'max_tokens': max_tokens}
    client = get_deploy_client('databricks') # conexión a model serving
    resp = client.predict(endpoint=endpoint, inputs=payload)
    return extract_chat_content(resp) # extraoig el texto en vez del json completo

def call_llm_simple(system_prompt: str, user_prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},]
    
    return call_llm(LLM_ENDPOINT, messages, temperature = 0.2, max_tokens = 500)


##### 4.2 - SQL Tool

In [0]:
FORBIDDEN = ['drop', 'delete', 'update', 'insert', 'alter', 'truncate', ';', '--', '/*', '*/']

def clean_sql(sql: str) -> str:
    s = sql.strip()

    if s.startswith('```'):
        lines = s.splitlines()

        if len(lines) >= 1 and lines[0].strip().startswith('```'):
            lines = lines[1:]

        if len(lines) >= 1 and lines[-1].strip().startswith('```'):
            lines = lines[:-1]

        s = '\n'.join(lines).strip()

    return s

def enforce_limit(sql: str, n: int = 20) -> str:
    s = sql.strip().rstrip(';')
    if 'limit' not in s.lower():
        return s + f'\nLIMIT {n}'
    return s

def validate_sql(sql: str, table: str):
    s = ' ' + sql.strip().lower() + ' '

    for bad in FORBIDDEN:
        if bad in s:
            raise ValueError(f'SQL bloqueada: {bad.strip()}')

    if table.lower() not in s:
        raise ValueError(f'La query SQL debe referenciar {table}')

    if not s.strip().startswith('select'):
        raise ValueError('La salida del modelo no parece una SELECT válida.')


In [0]:
def text_to_sql(question: str, table: str, schema_txt: str) -> str:
    system = 'Sos experto en SQL Spark (Databricks). Devolvé SOLO SQL. Nada de explicación.'
    user = f"""
Generá una query SQL Spark para responder la pregunta usando SOLO esta tabla: {table}

Esquema:
{schema_txt}

Reglas:
- Usá nombres exactos de columnas del esquema.
- No uses DROP/DELETE/UPDATE/INSERT/ALTER/TRUNCATE.
- Para filtros por texto (cuenta_bt, cuit, cliente, producto, sub_producto, moneda y oficial), siempre usá búsqueda PARCIAL case-insensitive:
   Usá: contains(columna, lower(trim('valor'))) = True
   Ejemplo: contains(cliente, lower(trim('Santander'))) = True
- No uses '=' para filtrar valores de texto.
- Cuando se pregunte por "resultado neto", "resultado bruto" o "IIBB", se tiene que hacer una suma por ese campo.
- Si preguntan por valor de un campo como "tasa activa" en los filtros se deben excluir registros null para ese campo:
  Ejemplo: Si buscan "tasa activa", usá: tasa_activa is not null
- Devolvé SOLO el SQL (sin ```).

Pregunta: {question}
"""

    raw = call_llm_simple(system, user)   
    sql = clean_sql(raw)                # limpia sql
    sql = enforce_limit(sql, 20)                  
    validate_sql(sql, table)              # valida
    return sql

#### 5.- Genero un router (a pulir con la práctica)

In [0]:
HINTS_SQL = ['matriz', 'ingresos', 'promedio', 'oficial', 'cliente','detalle', 'detalle por cliente', 'detalle por oficial',
             'CUIT', 'nombre del cliente', 'nombre del oficial', 'apellido del cliente','apellido del oficial']

HINTS_PDF = ['PPT', 'en el pdf', 'en el informe', 'en la presentación','promedio','evolución','suma','resultado',
             'P&L','negocio','banca', 'segmento','anual','YTD','monto mensual']

In [0]:
def route(question: str) -> str:
    q = question.lower()
    has_sql = any(k in q for k in HINTS_SQL)
    has_pdf = any(k in q for k in HINTS_PDF)

    if has_pdf and not has_sql:
        return 'pdf'
    if has_sql and not has_pdf:
        return 'sql'
    if has_sql and has_pdf:
        return 'hybrid'
    return 'sql'


#### 6.- Respuesta SQL

In [0]:
def answer_sql(question: str) -> str:
    sql = text_to_sql(question, TABLE_, SCHEMA_TEXT)

    print('---SQL GENERADA---')
    print(sql)
    print('------------------')

    r = run_sql_preview(sql, n=20)
    if not r['ok']:
        return f'No pude ejecutar la SQL generada.\nError: {r["error"]}\nSQL:\n{sql}'

    if not r['has_rows']:
        return f'La SQL se ejecutó pero no devolvió filas.\nSQL:\n{sql}'

    system = 'Sos un asistente de datos. Respondé en español. No inventes. Si falta info, decilo.'
    prompt = f'''
        Pregunta: {question}

        Evidencia SQL [T1]:
        SQL:
        {sql}

        Resultado:
        {r["preview"]}

        Instrucciones:
        - Usá [T1] para justificar números.
        - No inventes nada fuera del resultado.
    '''
    
    return call_llm_simple(system, prompt)

### 7.- Respuesta PDF

In [0]:
def answer_pdf(question: str) -> str:
    return 'fALTA'

### 8.- Respuesta Final

In [0]:
def answer(question: str) -> str:
    r = route(question)
    print('ROUTE:', r)

    if r == 'sql':
        return answer_sql(question)

    if r == 'pdf':
        out = answer_sql(question)
        return out + '\n\n(Nota: esta pregunta parecía de PDF, falta agregar.)'

    out = answer_sql(question)
    return out + '\n\n(Nota: también podría complementar con PDFs, pero falta agreagr.)'


In [0]:
# print(answer_sql("Mostrame el resultado neto IIBB total por producto y moneda"))
# print(answer_sql('Qué productos y subproductos tiene el cliente AB CONSTRUCCIONES SRL?'))
# print(answer_sql('Qué clientes maneja la oficial TROVATO MARIA SILVINA?'))
# print(answer_sql('Qué clientes maneja la oficial TROVATO Maria Silvina?'))

# print(answer_sql('Dame la tasa activa para el cliente santander para julio 2025'))
# print(answer_sql('Dame el volumen promedio para el cliente grimoldi para julio 2025'))
# print(answer_sql('Dame el resultado neto para el cliente grimoldi para julio 2025'))
# print(answer_sql('Dame el resultado neto de santander para julio 2025'))
# print(answer_sql('Dame el resultado neto de santander para julio 2025'))
# print(answer_sql('Dame el resultado neto de santander por producto para julio 2025'))
# print(answer_sql('Cuales subproductos distintos tiene santander para julio 2025'))

print(answer_sql("¿Cuál es el dato de previsiones para octubre 2025?"))